# Notebook 04 — Leakage-Safe Target & Dataset Split

## Patient Similarity Network + Agent-Based Modelling for Readmission

### Purpose
1. Define exactly what "readmission" means for this project.
2. Create a patient-level target.
3. Split patients into train / validation / test partitions.
4. Automatically verify zero patient overlap.

### Important dataset limitation
The Diabetes 130-US Hospitals dataset does not provide reliable encounter dates/timestamps.
Therefore, this notebook does **not** invent chronology from CSV row order.

A reproducible patient-level stratified split is used. The temporal limitation is explicitly
recorded, because a true temporal validation and strict pre-prediction feature reconstruction
cannot be verified from this dataset alone.

The project proposal requires patient-level splitting, zero-overlap assertions, and temporal
validation where timestamps support it.

## Project structure

```text
sna/
├── diabetes+130-us+hospitals+for+years+1999-2008/
├── notebooks/
│   ├── 01_dataset_audit.ipynb
│   ├── 02_cleaning.ipynb
│   ├── 03_patient_representation.ipynb
│   └── 04_split_leakage_check.ipynb
├── results/
└── figures/
```

### Inputs
- `results/03_patient_representation.csv`
- `results/03_target_summary_not_for_features.csv`

### Outputs
- train / validation / test patient tables
- train / validation / test patient-ID files
- split statistics
- target definition
- leakage audit
- reproducibility summary

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
TRAIN_SIZE = 0.70
VALID_SIZE = 0.15
TEST_SIZE = 0.15

# Primary target: observed readmission within 30 days.
PRIMARY_TARGET = "any_observed_readmission_under_30d"

# Sensitivity analysis target: any observed readmission (<30 or >30).
SENSITIVITY_TARGET = "any_observed_readmission"

pd.set_option("display.max_columns", 100)

cwd = Path.cwd()
candidate_roots = [cwd, cwd.parent, cwd.parent.parent]

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / "results" / "03_patient_representation.csv").exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find results/03_patient_representation.csv. "
        "Open this notebook inside the sna project folder."
    )

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PATIENT_PATH = RESULTS_DIR / "03_patient_representation.csv"
TARGET_PATH = RESULTS_DIR / "03_target_summary_not_for_features.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PATIENT INPUT:", PATIENT_PATH)
print("TARGET INPUT:", TARGET_PATH)

PROJECT_ROOT: c:\Users\Gayatri\OneDrive\Desktop\sna
PATIENT INPUT: c:\Users\Gayatri\OneDrive\Desktop\sna\results\03_patient_representation.csv
TARGET INPUT: c:\Users\Gayatri\OneDrive\Desktop\sna\results\03_target_summary_not_for_features.csv


In [2]:
patient_df = pd.read_csv(PATIENT_PATH, low_memory=False)
target_df = pd.read_csv(TARGET_PATH, low_memory=False)

print("Patient representation:", patient_df.shape)
print("Target summary:", target_df.shape)

assert "patient_nbr" in patient_df.columns
assert "patient_nbr" in target_df.columns
assert PRIMARY_TARGET in target_df.columns
assert SENSITIVITY_TARGET in target_df.columns

assert patient_df["patient_nbr"].is_unique
assert target_df["patient_nbr"].is_unique

print("Unique patients:", patient_df["patient_nbr"].nunique())

Patient representation: (71518, 39)
Target summary: (71518, 6)
Unique patients: 71518


# 1. Target definition

The source `readmitted` field contains:

- `NO` — no readmission
- `<30` — readmission within 30 days
- `>30` — readmission after 30 days

## Primary project target

A patient is positive when at least one of their observed encounters is labelled `<30`.

```text
PRIMARY_TARGET = any_observed_readmission_under_30d
```

## Sensitivity target

For sensitivity analysis, a patient is positive when at least one observed encounter is
labelled either `<30` or `>30`.

```text
SENSITIVITY_TARGET = any_observed_readmission
```

The sensitivity target does not replace the primary target.

### Unit of analysis
The target is patient-level because the project uses patients as nodes in the clinical
similarity network and Notebook 03 produced one row per patient.

This is an **observed patient-level outcome summary**. It is not claimed to be a prospective
prediction after a specific future discharge.

In [3]:
target_table = target_df[
    [
        "patient_nbr",
        "encounters_no_readmission",
        "encounters_readmission_over_30d",
        "encounters_readmission_under_30d",
        PRIMARY_TARGET,
        SENSITIVITY_TARGET,
    ]
].copy()

for col in [PRIMARY_TARGET, SENSITIVITY_TARGET]:
    values = set(target_table[col].dropna().unique())
    assert values.issubset({0, 1}), f"Unexpected values in {col}: {values}"
    target_table[col] = target_table[col].astype(int)

analysis_df = patient_df.merge(
    target_table,
    on="patient_nbr",
    how="inner",
    validate="one_to_one"
)

assert len(analysis_df) == len(patient_df)

print("Merged analytical table:", analysis_df.shape)

print("\nPrimary target distribution:")
print(analysis_df[PRIMARY_TARGET].value_counts().sort_index())

print("\nSensitivity target distribution:")
print(analysis_df[SENSITIVITY_TARGET].value_counts().sort_index())

Merged analytical table: (71518, 44)

Primary target distribution:
any_observed_readmission_under_30d
0    62684
1     8834
Name: count, dtype: int64

Sensitivity target distribution:
any_observed_readmission
0    42734
1    28784
Name: count, dtype: int64


# 2. Target and feature leakage audit

Outcome variables are kept outside the patient representation.

These columns are outcome-only:

- `encounters_no_readmission`
- `encounters_readmission_over_30d`
- `encounters_readmission_under_30d`
- `any_observed_readmission_under_30d`
- `any_observed_readmission`

They must never be supplied as similarity features or model predictors.

In [4]:
OUTCOME_ONLY_COLS = [
    "encounters_no_readmission",
    "encounters_readmission_over_30d",
    "encounters_readmission_under_30d",
    "any_observed_readmission_under_30d",
    "any_observed_readmission",
]

leakage_columns = sorted(
    set(patient_df.columns).intersection(OUTCOME_ONLY_COLS)
)

print("Outcome columns found in patient representation:", leakage_columns)

assert leakage_columns == [], "Outcome leakage detected."
assert "readmitted" not in patient_df.columns

Outcome columns found in patient representation: []


# 3. Temporal feasibility audit

The dataset has no reliable encounter date/timestamp.

We therefore **do not**:
- sort encounters by row number and call that time;
- claim that the last CSV row is the latest encounter;
- call all aggregated utilization "prior utilization";
- perform a fake temporal train/test split.

The limitation is saved as part of the reproducibility record and must be disclosed in the
final research paper.

In [5]:
timestamp_like_cols = [
    c for c in analysis_df.columns
    if any(token in c.lower() for token in ["date", "timestamp"])
]

temporal_audit = {
    "timestamp_columns_detected": timestamp_like_cols,
    "temporal_split_used": False,
    "csv_row_order_used_as_time": False,
    "reason": (
        "No reliable encounter date/timestamp is available; CSV row order is not treated "
        "as chronology."
    ),
    "strict_pre_prediction_feature_leakage_verifiable": False,
}

print(json.dumps(temporal_audit, indent=2))

{
  "timestamp_columns_detected": [],
  "temporal_split_used": false,
  "csv_row_order_used_as_time": false,
  "reason": "No reliable encounter date/timestamp is available; CSV row order is not treated as chronology.",
  "strict_pre_prediction_feature_leakage_verifiable": false
}


# 4. Patient-level train / validation / test split

Because timestamps do not support a temporal split, the primary reproducible split is:

- 70% train
- 15% validation
- 15% test

The split is performed on **unique patients**, not encounter rows.

Stratification uses the primary `<30` target.

In [6]:
patients = analysis_df[["patient_nbr", PRIMARY_TARGET]].copy()

train_patients, temp_patients = train_test_split(
    patients,
    test_size=VALID_SIZE + TEST_SIZE,
    random_state=SEED,
    stratify=patients[PRIMARY_TARGET]
)

valid_patients, test_patients = train_test_split(
    temp_patients,
    test_size=TEST_SIZE / (VALID_SIZE + TEST_SIZE),
    random_state=SEED,
    stratify=temp_patients[PRIMARY_TARGET]
)

train_ids = set(train_patients["patient_nbr"])
valid_ids = set(valid_patients["patient_nbr"])
test_ids = set(test_patients["patient_nbr"])

print("Train patients:", len(train_ids))
print("Validation patients:", len(valid_ids))
print("Test patients:", len(test_ids))
print("Total:", len(train_ids) + len(valid_ids) + len(test_ids))

Train patients: 50062
Validation patients: 10728
Test patients: 10728
Total: 71518


# 5. HARD zero-overlap check

This is a mandatory correctness condition.

No patient may appear in more than one partition.

In [7]:
overlap_train_valid = train_ids.intersection(valid_ids)
overlap_train_test = train_ids.intersection(test_ids)
overlap_valid_test = valid_ids.intersection(test_ids)

all_original_ids = set(analysis_df["patient_nbr"])
all_split_ids = train_ids | valid_ids | test_ids

print("Train ∩ Validation:", len(overlap_train_valid))
print("Train ∩ Test:", len(overlap_train_test))
print("Validation ∩ Test:", len(overlap_valid_test))
print("Missing patients:", len(all_original_ids - all_split_ids))
print("Unexpected patients:", len(all_split_ids - all_original_ids))

assert len(overlap_train_valid) == 0
assert len(overlap_train_test) == 0
assert len(overlap_valid_test) == 0
assert all_split_ids == all_original_ids

print("\nZERO-OVERLAP CHECK: PASS")

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
Missing patients: 0
Unexpected patients: 0

ZERO-OVERLAP CHECK: PASS


# 6. Split statistics

Check patient counts and target prevalence in every partition.

Small differences are expected because patient counts must be integers.

In [8]:
train_full = analysis_df[analysis_df["patient_nbr"].isin(train_ids)].copy()
valid_full = analysis_df[analysis_df["patient_nbr"].isin(valid_ids)].copy()
test_full = analysis_df[analysis_df["patient_nbr"].isin(test_ids)].copy()

def split_stats(name, frame):
    return {
        "split": name,
        "n_patients": int(len(frame)),
        "positive_primary": int(frame[PRIMARY_TARGET].sum()),
        "primary_positive_rate": float(frame[PRIMARY_TARGET].mean()),
        "positive_sensitivity": int(frame[SENSITIVITY_TARGET].sum()),
        "sensitivity_positive_rate": float(frame[SENSITIVITY_TARGET].mean()),
    }

split_stats_df = pd.DataFrame([
    split_stats("train", train_full),
    split_stats("validation", valid_full),
    split_stats("test", test_full),
])

overall_rate = float(analysis_df[PRIMARY_TARGET].mean())
split_stats_df["absolute_difference_from_overall"] = (
    split_stats_df["primary_positive_rate"] - overall_rate
).abs()

display(split_stats_df)
print("Overall primary positive rate:", round(overall_rate, 4))

,split,n_patients,positive_primary,primary_positive_rate,positive_sensitivity,sensitivity_positive_rate,absolute_difference_from_overall
0,train,50062,6184,0.123527,20095,0.401402,0.000005
1,validation,10728,1325,0.123509,4321,0.402778,0.000013
2,test,10728,1325,0.123509,4368,0.407159,0.000013


Overall primary positive rate: 0.1235


# 7. Save split tables and immutable patient-ID lists

The test set created here must not be used for model fitting, feature selection, ABM calibration,
or intervention tuning.

Later notebooks should load these ID files rather than silently creating a new split.

In [9]:
TRAIN_OUTPUT = RESULTS_DIR / "04_train_patients.csv"
VALID_OUTPUT = RESULTS_DIR / "04_validation_patients.csv"
TEST_OUTPUT = RESULTS_DIR / "04_test_patients.csv"

TRAIN_IDS_OUTPUT = RESULTS_DIR / "04_train_patient_ids.csv"
VALID_IDS_OUTPUT = RESULTS_DIR / "04_validation_patient_ids.csv"
TEST_IDS_OUTPUT = RESULTS_DIR / "04_test_patient_ids.csv"

train_full.to_csv(TRAIN_OUTPUT, index=False)
valid_full.to_csv(VALID_OUTPUT, index=False)
test_full.to_csv(TEST_OUTPUT, index=False)

pd.DataFrame({"patient_nbr": sorted(train_ids)}).to_csv(TRAIN_IDS_OUTPUT, index=False)
pd.DataFrame({"patient_nbr": sorted(valid_ids)}).to_csv(VALID_IDS_OUTPUT, index=False)
pd.DataFrame({"patient_nbr": sorted(test_ids)}).to_csv(TEST_IDS_OUTPUT, index=False)

print("Saved:")
for p in [
    TRAIN_OUTPUT, VALID_OUTPUT, TEST_OUTPUT,
    TRAIN_IDS_OUTPUT, VALID_IDS_OUTPUT, TEST_IDS_OUTPUT
]:
    print(" -", p)

Saved:
 - c:\Users\Gayatri\OneDrive\Desktop\sna\results\04_train_patients.csv
 - c:\Users\Gayatri\OneDrive\Desktop\sna\results\04_validation_patients.csv
 - c:\Users\Gayatri\OneDrive\Desktop\sna\results\04_test_patients.csv
 - c:\Users\Gayatri\OneDrive\Desktop\sna\results\04_train_patient_ids.csv
 - c:\Users\Gayatri\OneDrive\Desktop\sna\results\04_validation_patient_ids.csv
 - c:\Users\Gayatri\OneDrive\Desktop\sna\results\04_test_patient_ids.csv


# 8. Final checkpoint — Notebook 04

### GO criteria

- Primary target is explicitly defined.
- Patient count is preserved.
- Train / validation / test contain unique patients.
- Every pair of partitions has zero patient overlap.
- Every patient is assigned exactly once.
- Outcome columns are absent from the feature representation.
- Split files are saved.
- The temporal limitation is explicitly documented.

### STOP

Stop if any patient overlap exists, patient counts change unexpectedly, target coding is wrong,
or outcome variables enter the feature representation.

In [10]:
final_checks = {
    "primary_target_defined": PRIMARY_TARGET in target_df.columns,
    "patient_count_preserved": len(analysis_df) == patient_df["patient_nbr"].nunique(),
    "train_ids_unique": len(train_ids) == len(train_patients),
    "validation_ids_unique": len(valid_ids) == len(valid_patients),
    "test_ids_unique": len(test_ids) == len(test_patients),
    "train_validation_zero_overlap": len(overlap_train_valid) == 0,
    "train_test_zero_overlap": len(overlap_train_test) == 0,
    "validation_test_zero_overlap": len(overlap_valid_test) == 0,
    "all_patients_assigned_once": all_split_ids == all_original_ids,
    "no_outcome_columns_in_features": leakage_columns == [],
    "train_output_exists": TRAIN_OUTPUT.exists(),
    "validation_output_exists": VALID_OUTPUT.exists(),
    "test_output_exists": TEST_OUTPUT.exists(),
    "train_id_output_exists": TRAIN_IDS_OUTPUT.exists(),
    "validation_id_output_exists": VALID_IDS_OUTPUT.exists(),
    "test_id_output_exists": TEST_IDS_OUTPUT.exists(),
    "temporal_limitation_documented": temporal_audit["strict_pre_prediction_feature_leakage_verifiable"] is False,
}

print("=" * 75)
print("NOTEBOOK 04 — FINAL CHECKPOINT")
print("=" * 75)

for name, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL':<6} | {name}")

print("=" * 75)

if all(final_checks.values()):
    print("OVERALL RESULT: PASS")
    print("Proceed to MANUAL REVIEW.")
else:
    print("OVERALL RESULT: FAIL")
    print("Fix the failed checks before moving to Notebook 05.")

NOTEBOOK 04 — FINAL CHECKPOINT
PASS   | primary_target_defined
PASS   | patient_count_preserved
PASS   | train_ids_unique
PASS   | validation_ids_unique
PASS   | test_ids_unique
PASS   | train_validation_zero_overlap
PASS   | train_test_zero_overlap
PASS   | validation_test_zero_overlap
PASS   | all_patients_assigned_once
PASS   | no_outcome_columns_in_features
PASS   | train_output_exists
PASS   | validation_output_exists
PASS   | test_output_exists
PASS   | train_id_output_exists
PASS   | validation_id_output_exists
PASS   | test_id_output_exists
PASS   | temporal_limitation_documented
OVERALL RESULT: PASS
Proceed to MANUAL REVIEW.


In [11]:
SUMMARY_OUTPUT = RESULTS_DIR / "04_split_leakage_summary.json"

summary = {
    "notebook": "04_split_leakage_check",
    "seed": int(SEED),
    "split_ratio": {
        "train": float(TRAIN_SIZE),
        "validation": float(VALID_SIZE),
        "test": float(TEST_SIZE)
    },
    "primary_target": PRIMARY_TARGET,
    "sensitivity_target": SENSITIVITY_TARGET,
    "primary_target_definition": (
        "1 if a patient has at least one observed encounter labelled <30; otherwise 0."
    ),
    "temporal_split_used": bool(temporal_audit["temporal_split_used"]),
    "temporal_audit": temporal_audit,
    "patient_counts": {
        "all": int(len(analysis_df)),
        "train": int(len(train_ids)),
        "validation": int(len(valid_ids)),
        "test": int(len(test_ids))
    },
    "final_checks": {str(k): bool(v) for k, v in final_checks.items()}
}

SUMMARY_OUTPUT.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("Summary saved:", SUMMARY_OUTPUT)

Summary saved: c:\Users\Gayatri\OneDrive\Desktop\sna\results\04_split_leakage_summary.json
